# 01 – Export W4A4 MicroCNN → FINN-ONNX
**FINN Pipeline Step 2/4**


## Cell 1 — Imports + Model + Load

In [15]:
import torch
import pathlib

# 1. Setup the directory (Using the current folder since you uploaded it here)
BUILD_DIR = pathlib.Path('.')
ckpt_path = BUILD_DIR / 'best_fashion_mobilenet.pth'

# 2. Load the raw weights directly 
state_dict = torch.load(ckpt_path, map_location='cpu')

print("--- NEW MODEL LAYER MAP ---")
print("Notice how the layers now match our 6-Layer CNN blueprint!\n")

# 3. Print a snippet to verify
for i, key in enumerate(state_dict.keys()):
    print(key)
    if i >= 12:
        print("... (truncating for space)")
        break

--- NEW MODEL LAYER MAP ---
Notice how the layers now match our 6-Layer CNN blueprint!

features.0.weight
features.1.weight
features.1.bias
features.1.running_mean
features.1.running_var
features.1.num_batches_tracked
features.2.act_quant.fused_activation_quant_proxy.tensor_quant.scaling_impl.value
features.4.weight
features.5.weight
features.5.bias
features.5.running_mean
features.5.running_var
features.5.num_batches_tracked
... (truncating for space)


In [13]:
print("\n--- FULL MODEL LAYER MAP ---")
# Let's print every key directly from our raw weights to see the exact naming
for key in state_dict.keys():
    print(key)


--- FULL MODEL LAYER MAP ---
features.0.weight
features.1.weight
features.1.bias
features.1.running_mean
features.1.running_var
features.1.num_batches_tracked
features.2.act_quant.fused_activation_quant_proxy.tensor_quant.scaling_impl.value
features.4.weight
features.5.weight
features.5.bias
features.5.running_mean
features.5.running_var
features.5.num_batches_tracked
features.6.act_quant.fused_activation_quant_proxy.tensor_quant.scaling_impl.value
features.8.weight
features.9.weight
features.9.bias
features.9.running_mean
features.9.running_var
features.9.num_batches_tracked
features.10.act_quant.fused_activation_quant_proxy.tensor_quant.scaling_impl.value
features.12.weight
features.13.weight
features.13.bias
features.13.running_mean
features.13.running_var
features.13.num_batches_tracked
features.14.act_quant.fused_activation_quant_proxy.tensor_quant.scaling_impl.value
classifier.0.weight


In [16]:
print("--- CONV WEIGHT SHAPES ---")
# Dynamically search for and print the shapes of all weight layers
for key, tensor in state_dict.items():
    if 'weight' in key:
        print(f"{key}: {tensor.shape}")

--- CONV WEIGHT SHAPES ---
features.0.weight: torch.Size([64, 1, 3, 3])
features.1.weight: torch.Size([64])
features.4.weight: torch.Size([64, 64, 3, 3])
features.5.weight: torch.Size([64])
features.8.weight: torch.Size([128, 64, 3, 3])
features.9.weight: torch.Size([128])
features.12.weight: torch.Size([128, 128, 3, 3])
features.13.weight: torch.Size([128])
classifier.0.weight: torch.Size([10, 512])


## Cell 2 — Export QONNX

In [17]:
import os, pathlib
import torch, torch.nn as nn
import brevitas.nn as qnn
from brevitas.export import export_qonnx
from qonnx.util.cleanup import cleanup as qonnx_cleanup

# 1. MATCHING ARCHITECTURE
class Fashion_Fit_CNN(nn.Module):
    def __init__(self, in_channels=1, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            qnn.QuantConv2d(in_channels, 64, 3, padding=1, bias=False, weight_bit_width=8),
            nn.BatchNorm2d(64),
            qnn.QuantReLU(bit_width=8),
            nn.MaxPool2d(2),
            qnn.QuantConv2d(64, 64, 3, padding=1, bias=False, weight_bit_width=2),
            nn.BatchNorm2d(64),
            qnn.QuantReLU(bit_width=2),
            nn.MaxPool2d(2),
            qnn.QuantConv2d(64, 128, 3, padding=1, bias=False, weight_bit_width=2),
            nn.BatchNorm2d(128),
            qnn.QuantReLU(bit_width=2),
            nn.MaxPool2d(2),
            qnn.QuantConv2d(128, 128, 3, padding=1, bias=False, weight_bit_width=2),
            nn.BatchNorm2d(128),
            qnn.QuantReLU(bit_width=2),
            nn.MaxPool2d(2) 
        )
        self.classifier = nn.Sequential(
            qnn.QuantLinear(512, num_classes, bias=False, weight_bit_width=8)
        )
    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x

# 2. LOAD WEIGHTS
DEVICE = torch.device('cpu')
model = Fashion_Fit_CNN(in_channels=1, num_classes=10).to(DEVICE)
model.load_state_dict(state_dict) # Loads your new .pth
model.eval()

# 3. EXPORT 32x32 TENSOR
DUMMY = torch.randn(1, 1, 32, 32)
QONNX_PATH = str(BUILD_DIR / 'fashion_cnn_qonnx.onnx')
QONNX_CLEAN = str(BUILD_DIR / 'fashion_cnn_qonnx_clean.onnx')

export_qonnx(model, DUMMY, QONNX_PATH)
qonnx_cleanup(QONNX_PATH, out_file=QONNX_CLEAN)
print(f"✅ Exported 4-Layer CNN to: {QONNX_CLEAN}")

✅ Exported 4-Layer CNN to: fashion_cnn_qonnx_clean.onnx


## Cell 3 — Verify QONNX

In [21]:
!ls /home/yeager/finn/notebooks/claude_builds/ver_2_w4_a4/

00_train_cifar10_w4a4.ipynb    cat.png	   microcnn_best.pt
01_export_finn.ipynb	       deer.png    microcnn_qonnx_clean.onnx
02_hardware_build_fixed.ipynb  dog.png	   microcnn_qonnx.onnx
02_hardware_build.ipynb        flight.png
03_pynq_deploy.ipynb	       frog.png


In [18]:
import numpy as np
import torch
from qonnx.core.modelwrapper import ModelWrapper
import qonnx.core.onnx_exec as oxe

# 🌟 THE FIX: Calculate the PyTorch reference output right here
with torch.no_grad():
    ref_out = model(DUMMY).detach().cpu().numpy()

# 1. Load the exported ONNX model
qonnx_model = ModelWrapper(QONNX_CLEAN)
inp_name = qonnx_model.graph.input[0].name

# 2. Execute the ONNX model using the FINN simulator
qonnx_out = oxe.execute_onnx(qonnx_model, {inp_name: DUMMY.numpy()})
qonnx_pred = list(qonnx_out.values())[0]

# 3. Compare PyTorch vs ONNX
match = np.allclose(ref_out, qonnx_pred, atol=1e-3)

print("--- VERIFICATION RESULTS ---")
print(f"QONNX matches PyTorch float: {match}")
if match:
    print("✅ PASS: Your 4-Layer model exported perfectly. Proceed to Hardware Build!")
else:
    print("❌ FAIL: Math mismatch detected.")

--- VERIFICATION RESULTS ---
QONNX matches PyTorch float: True
✅ PASS: Your 4-Layer model exported perfectly. Proceed to Hardware Build!


## Cell 4 — QONNX → FINN-ONNX + Preprocessing

In [19]:
from finn.util.pytorch import ToTensor
from finn.util.visualization import showInNetron
from qonnx.transformation.merge_onnx_models import MergeONNXModels
from qonnx.core.datatype import DataType
from finn.transformation.qonnx.convert_qonnx_to_finn import ConvertQONNXtoFINN
from qonnx.transformation.infer_shapes import InferShapes
from qonnx.transformation.fold_constants import FoldConstants
from qonnx.transformation.general import GiveReadableTensorNames, GiveUniqueNodeNames, RemoveStaticGraphInputs
from qonnx.transformation.infer_datatypes import InferDataTypes
from qonnx.transformation.insert_topk import InsertTopK

FINN_TIDY = str(BUILD_DIR / 'fashion_cnn_finn_tidy.onnx')

# ⚠️ SHRUNK TO 32x32!
preproc_path = str(BUILD_DIR / 'preproc.onnx')
totensor = ToTensor()
export_qonnx(totensor, torch.zeros(1, 1, 32, 32), preproc_path)
qonnx_cleanup(preproc_path, out_file=preproc_path)
pre_model = ModelWrapper(preproc_path)
pre_model = pre_model.transform(ConvertQONNXtoFINN())

# Merge the preprocessor with our main model
main_model = ModelWrapper(QONNX_CLEAN)
main_model = main_model.transform(ConvertQONNXtoFINN())
main_model = main_model.transform(MergeONNXModels(pre_model))

inp_name = main_model.graph.input[0].name
main_model.set_tensor_datatype(inp_name, DataType['UINT8'])

# Add Top-1 postprocessing
main_model = main_model.transform(InsertTopK(k=1))

# Tidy up the graph structure
main_model = main_model.transform(InferShapes())
main_model = main_model.transform(FoldConstants())
main_model = main_model.transform(GiveUniqueNodeNames())
main_model = main_model.transform(GiveReadableTensorNames())
main_model = main_model.transform(InferDataTypes())
main_model = main_model.transform(RemoveStaticGraphInputs())

main_model.save(FINN_TIDY)
print(f"✅ FINN-ONNX saved -> {FINN_TIDY}")

# Optional: Show in Netron
# showInNetron(FINN_TIDY)

✅ FINN-ONNX saved -> fashion_cnn_finn_tidy.onnx


/home/yeager/finn/deps/qonnx/src/qonnx/transformation/infer_data_layouts.py:127: UserWarning: Assuming 4D input is NCHW
  warnings.warn("Assuming 4D input is NCHW")


## Cell 5 — Verify All Files

In [9]:
print("Files in BUILD_DIR:")
for f in sorted(BUILD_DIR.iterdir()):
    sz = f.stat().st_size/1024
    print(f"  {'✅' if sz>0 else '❌'}  {f.name:45s}  {sz:7.1f} KB")
print("\n✅ Ready for 02_hardware_build.ipynb")

Files in BUILD_DIR:
  ✅  .ipynb_checkpoints                                 4.0 KB
  ✅  01_export_finn.ipynb                              19.4 KB
  ✅  02_hardware_build_fixed.ipynb                     42.7 KB
  ✅  best_fashion_mobilenet.pth                      1082.6 KB
  ✅  fashion_cnn_finn_tidy.onnx                      1113.8 KB
  ✅  fashion_cnn_qonnx.onnx                          1050.8 KB
  ✅  fashion_cnn_qonnx_clean.onnx                    1045.3 KB
  ✅  preproc.onnx                                       0.2 KB

✅ Ready for 02_hardware_build.ipynb
